# Countries and Anime: The Global Geography of Taste

## 1. Introduction to Global Analysis
Japanese animation (anime) has now transcended its original borders, establishing itself as a cultural phenomenon of global reach. The advent of streaming platforms and the increasing digitalization of fan communities have made this medium accessible in every corner of the world, generating a vast and incredibly diverse ecosystem of consumption.

This notebook aims to explore the intersection of **geography, demographics, and cultural consumption**. Leveraging a large-scale dataset containing millions of user interactions (ratings, preferences, and geographic profiling data), we will investigate how viewing habits and tastes are distributed globally.

The analysis will not simply passively map user density, but will seek to answer complex analytical questions: *
How does geographic location influence the perception of a work? Are there specific behavioral patterns or cultural biases in the severity of ratings?* Through a data-driven approach and the use of interactive visualizations, we will build a geographical storytelling that will allow us to understand the dynamics of a global community, extracting valuable insights from the massive amount of data at our disposal.

In [ ]:
import pandas as pd
import time
import gc

# --- SETUP ---
print("Initiating memory-optimized data integration process...")
start_time = time.time()

print("Step 1: Loading profile data...")
# Load strictly necessary columns to minimize memory footprint
profiles = pd.read_parquet('../data/profiles_clean.parquet', engine='pyarrow', columns=['username', 'location'])

# Create a fast lookup Series mapping 'username' to 'location'
# Casting 'location' to categorical type drastically reduces memory consumption
location_map = profiles.set_index('username')['location'].astype('category')

# Explicit memory deallocation: remove the profiles dataframe and force garbage collection
del profiles
gc.collect()

print("Step 2: Loading ratings data...")
user_ratings = pd.read_parquet('../data/ratings_clean.parquet', engine='pyarrow', columns=['username', 'anime_id', 'score'])

print("Step 3: Mapping geographic locations to user ratings...")
# Utilize map() as a highly optimized, lightweight alternative to pd.merge()
user_ratings['location'] = user_ratings['username'].map(location_map)

print("Step 4: Optimizing final dataset...")
# Drop the 'username' column (string/object dtype). This frees up gigabytes of RAM 
# and is no longer required for macro-geographic visual analysis.
user_ratings.drop(columns=['username'], inplace=True)

# Replicate 'inner join' behavior by dropping records that failed to map to a valid location
user_ratings.dropna(subset=['location'], inplace=True)

# --- EXECUTION SUMMARY ---
end_time = time.time()
execution_time = round(end_time - start_time, 2)

print("--------------------------------------------------")
print(f"Task completed successfully in {execution_time} seconds.")
print(f"Final dataset dimensions: {user_ratings.shape[0]:,} rows.")
print("--------------------------------------------------")

Initiating memory-optimized data integration process...
Step 1: Loading profile data...
Step 2: Loading ratings data...
Step 3: Mapping geographic locations to user ratings...


## 2. Data Integration Strategy & Memory Optimization

### Overview
This module manages the integration of large-scale datasets, merging user ratings (Ratings) with geographic profile data (Profiles). To overcome the hardware and memory limitations (MemoryErrors) typical of standard join operations on large data volumes, a dictionary-mapping strategy has been implemented, supported by active garbage collection management.

### Methodology and Technical Justification
Integrating large datasets (over 90 million records) presents critical challenges related to RAM management. The Pandas pd.merge() approach alone generates significant computational overhead, quickly saturating available memory due to inefficient handling of string objects (object dtypes).

To ensure system scalability, efficiency, and stability, we adopted the following process architecture:

* **Dtype Optimization:** The spatial column `location` has been converted to the type `category`. This dramatically reduces the memory footprint by storing textual data as integer indexes rather than repeated strings.
* **Dictionary-Mapping (Pseudo-Join):** Instead of performing a traditional *inner join* on tens of millions of rows, the profile data frame has been transformed into a fast-access mapping dictionary (`location_map`). The `map()` operation is significantly lighter and faster in terms of RAM allocation than an expensive relational `merge`.
* **Active Memory Management:** Explicit object lifecycle management has been implemented using the native `gc` (Garbage Collector) module. Immediately removing (de-allocating) the `profiles` dataframe once the mapping has been extracted frees up valuable resources before loading the massive ratings dataset.
* **Garbage Collection:** Invoking `gc.collect()` forces the operating system to release previously occupied but no longer used memory, radically mitigating the risk of crashes.

### Performance Considerations
The adoption of this memory-optimized algorithmic construct allows the entire dataset to be processed and merged in an extremely short amount of time. This timeframe represents an optimal trade-off in relation to the volume of data processed, ensuring the generation of a robust unified dataset without compromising local hardware limitations.

In [ ]:
import plotly.express as px

# Aggregate vote counts per location
country_counts = user_ratings['location'].value_counts().reset_index()
country_counts.columns = ['location', 'total_votes']

# Filter out missing or unknown locations
country_counts = country_counts[~country_counts['location'].isin(['Unknown', ''])]

# Extract top 20 countries
top_20_countries = country_counts.head(20)

# Generate interactive bar chart
fig = px.bar(
    top_20_countries,
    x='location',
    y='total_votes',
    title='Top 20 Countries by Total Anime Votes',
    labels={'location': 'Country', 'total_votes': 'Total Votes'},
    color='total_votes',
    color_continuous_scale='Sunsetdark',
    text_auto='.2s'
)

# Optimize layout for readability
fig.update_layout(
    xaxis_tickangle=-45,
    plot_bgcolor='rgba(0,0,0,0)',
    font=dict(size=14)
)

fig.show()